# Previsão de aprovação de empréstimos com Random Forest
Desafio Rocketseat — classificação de `loan_status` (1 = aprovado, 0 = rejeitado) com as 13 variáveis preditoras.

In [ ]:
import pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

## 1. Carregamento e exploração

In [ ]:
df = pd.read_csv("loan_data.csv")
print(df.shape, "| nulos:", df.isna().sum().sum(), "| duplicados:", df.duplicated().sum())
print(df.loan_status.value_counts(normalize=True))
df.describe().T

Observações: não há nulos, mas existem valores irreais (idade até 144 anos, experiência maior que a idade possível). As classes são desbalanceadas (~78% rejeitados / ~22% aprovados).

## 2. Limpeza

In [ ]:
n0 = len(df)
df = df[(df.person_age <= 100) & (df.person_emp_exp <= df.person_age - 14)].reset_index(drop=True)
print("Linhas removidas:", n0 - len(df))

## 3. Separação treino/teste e pré-processamento
Divisão estratificada 80/20. As variáveis categóricas recebem One-Hot Encoding dentro do `Pipeline`, evitando vazamento de dados. Random Forest não exige escalonamento.

In [ ]:
X, y = df.drop(columns="loan_status"), df["loan_status"]
cat = X.select_dtypes(exclude="number").columns.tolist()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat)], remainder="passthrough")
pipe = Pipeline([("pre", pre),
                 ("rf", RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"))])

## 4. Treino e validação cruzada
Busca aleatória de hiperparâmetros otimizando o **F1** (por causa do desbalanceamento). Para um ajuste mais fino, aumente `n_iter` e `n_estimators` (aqui o espaço é pequeno para rodar rápido).

In [ ]:
grid = {"rf__n_estimators": [100, 150],
        "rf__max_depth": [10, 15, 20],
        "rf__min_samples_leaf": [1, 2, 4],
        "rf__max_features": ["sqrt", "log2", 0.5]}
search = RandomizedSearchCV(pipe, grid, n_iter=5, scoring="f1",
                            cv=StratifiedKFold(3, shuffle=True, random_state=42),
                            random_state=42, n_jobs=-1)
search.fit(Xtr, ytr)
print(search.best_params_, round(search.best_score_, 4))

## 5. Avaliação no conjunto de teste

In [ ]:
best = search.best_estimator_
pred = best.predict(Xte); proba = best.predict_proba(Xte)[:, 1]
print(classification_report(yte, pred, target_names=["Rejeitado", "Aprovado"]))
print("ROC AUC:", round(roc_auc_score(yte, proba), 4))

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ConfusionMatrixDisplay.from_predictions(yte, pred, ax=ax[0], colorbar=False); ax[0].set_title("Matriz de confusão")
RocCurveDisplay.from_predictions(yte, proba, ax=ax[1]); ax[1].set_title("Curva ROC")
names = [n.split("__")[-1] for n in best.named_steps["pre"].get_feature_names_out()]
imp = pd.Series(best.named_steps["rf"].feature_importances_, index=names).sort_values().tail(12)
imp.plot.barh(ax=ax[2]); ax[2].set_title("Top 12 importâncias")
plt.tight_layout(); plt.show()

## 6. Conclusões
- O histórico de inadimplência anterior é o fator mais relevante, seguido de taxa de juros, renda e percentual da renda comprometido.
- Como os dados são sintéticos, os resultados não devem ser extrapolados para o mundo real.
- Próximos passos: testar outros modelos de ensemble (Gradient Boosting), ajustar o limiar de decisão e analisar a importância por permutação.

In [ ]:
joblib.dump(best, "modelo_random_forest.joblib")